In [ ]:
# ============================================================
# CNN - CONVOLUTIONAL NEURAL NETWORK
# ============================================================
# OBJECTIVE:
# Develop a CNN model that can classify fruit images
# as Fresh or Rotten.
# ============================================================


# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense
)


# ============================================================
# STEP 2: LOAD DATASET
# ============================================================
 

 
train_dataset = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(128, 128),
    batch_size=32,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    "test",
    image_size=(128, 128),
    batch_size=32,
)


# ============================================================
# STEP 3: CHECK CLASSES
# ============================================================

class_names = train_dataset.class_names

print("Fruit Classes:")
print(class_names)

print("Number of Classes:", len(class_names))


# ============================================================
# STEP 4: NORMALIZE IMAGES
# ============================================================

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

test_dataset = test_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)


# ============================================================
# STEP 5: DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])


# ============================================================
# STEP 6: BUILD CNN MODEL
# ============================================================

model = Sequential([

    data_augmentation,

    Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(128, 128, 3)
    ),

    MaxPooling2D((2, 2)),

    Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D((2, 2)),

    Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),

    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),

    Dense(
        6,
        activation="softmax"
    )
])


# ============================================================
# STEP 7: DISPLAY MODEL ARCHITECTURE
# ============================================================

model.summary()


# ============================================================
# STEP 8: COMPILE THE MODEL
# ============================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# STEP 9: TRAIN THE MODEL
# ============================================================

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)


# ============================================================
# STEP 10: EVALUATE THE MODEL
# ============================================================

loss, accuracy = model.evaluate(test_dataset)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


# ============================================================
# STEP 11: TRAINING LOSS VS VALIDATION LOSS
# ============================================================

plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend([
    "Training Loss",
    "Validation Loss"
])

plt.show()


# ============================================================
# STEP 12: PREDICT AN UNSEEN IMAGE
# ============================================================

img = tf.keras.utils.load_img(
    "test_image.png",
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

prediction = model.predict(img_array)

predicted_class = class_names[
    np.argmax(prediction)
]

confidence = np.max(prediction) * 100

print("Prediction:", predicted_class)
print("Confidence:", confidence, "%")


# ============================================================
# STEP 13: CHECK TEST FOLDERS
# ============================================================

import os

test_path = "test"

for folder in os.listdir(test_path):
    print(folder)


# ============================================================
# STEP 14: GET AN ACTUAL TEST IMAGE
# ============================================================

image_path = None

for root, dirs, files in os.walk("test"):

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            image_path = os.path.join(
                root,
                file
            )

            break

    if image_path:
        break

print("Image:", image_path)


# ============================================================
# STEP 15: PREDICT THE ACTUAL TEST IMAGE
# ============================================================

img = tf.keras.utils.load_img(
    image_path,
    target_size=(128, 128)
)

img_array = tf.keras.utils.img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

prediction = model.predict(img_array)

predicted_class = class_names[
    np.argmax(prediction)
]

confidence = np.max(prediction) * 100

print("Prediction:", predicted_class)

print(
    "Confidence:",
    round(confidence, 2),
    "%"
)